# MSBD 566 – Challenging Data Small Task

**Name:** David Reyes  
**Course:** MSBD 566  
**Date:** November 6, 2026

I checked the same BRFSS random forest with shuffled labels and reported uncertainty and the event rate for Lecture 12.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv('diabetes_012_health_indicators_BRFSS2015.csv')
X = df.drop(columns='Diabetes_012')
y = (df['Diabetes_012'] > 0).astype(int)
print('Data shape:', df.shape)

X_work, X_test, y_work, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_work, y_work, test_size=0.20, stratify=y_work, random_state=42)

Data shape: (253680, 22)


## 1. Train And Report The Base Rate

I used the same random forest and validation split as my earlier task. The event rate is the fraction reporting prediabetes or diabetes. Average precision measures how well the model finds positive cases across cutoffs; the event rate is its reference level for random ranking.

In [2]:
rf = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
scores = rf.predict_proba(X_valid)[:, 1]

base_rate = y_valid.mean()
auc = roc_auc_score(y_valid, scores)
ap = average_precision_score(y_valid, scores)
print('Validation responses:', len(y_valid))
print('Recorded positive cases:', int(y_valid.sum()))
display(pd.Series({'Event rate': base_rate, 'Average precision': ap,
                   'ROC-AUC': auc}).round(4))

Validation responses: 40589
Recorded positive cases: 6396


Event rate           0.1576
Average precision    0.4593
ROC-AUC              0.8245
dtype: float64

## 2. Shuffle The Labels

I shuffled the outcomes separately within training and validation, then fitted a fresh copy of the whole model. The model only uses the original numeric inputs, with no extra preprocessing, tuning, or feature selection. An AUC near 0.50 is expected after shuffling. A small difference from 0.50 can happen by chance; a clearly high result would need investigation for leakage.

In [3]:
shuffled_train = y_train.sample(frac=1, random_state=42).to_numpy()
shuffled_valid = y_valid.sample(frac=1, random_state=43).to_numpy()

shuffled_rf = clone(rf)
shuffled_rf.fit(X_train, shuffled_train)
shuffled_scores = shuffled_rf.predict_proba(X_valid)[:, 1]
shuffled_auc = roc_auc_score(shuffled_valid, shuffled_scores)
shuffled_ap = average_precision_score(shuffled_valid, shuffled_scores)

display(pd.DataFrame({
    'ROC-AUC': [auc, shuffled_auc, 0.50],
    'Average precision': [ap, shuffled_ap, base_rate]
}, index=['Original labels', 'Shuffled labels', 'Random-ranking reference']).round(4))

,ROC-AUC,Average precision
Original labels,0.8245,0.4593
Shuffled labels,0.4985,0.1565
Random-ranking reference,0.5000,0.1576


## 3. Bootstrap An Interval

The slide requests a patient bootstrap. This file has no patient ID, so I used a row bootstrap as a fallback, not a patient bootstrap. I sampled validation rows with replacement 500 times and used the middle 95% of scores as a percentile interval. Each sampled row keeps its outcome and prediction together.

This estimates validation-sample uncertainty for this fixed model. It does not include model-retraining variation or account for unknown repeated people. If person IDs become available, resample whole people and split by person instead.

In [4]:
rng = np.random.default_rng(42)
labels = y_valid.to_numpy()
bootstrap_auc = []
bootstrap_ap = []

for repeat in range(500):
    rows = rng.integers(0, len(labels), size=len(labels))
    bootstrap_auc.append(roc_auc_score(labels[rows], scores[rows]))
    bootstrap_ap.append(average_precision_score(labels[rows], scores[rows]))

intervals = pd.DataFrame({
    'Estimate': [auc, ap],
    '95% lower': [np.percentile(bootstrap_auc, 2.5), np.percentile(bootstrap_ap, 2.5)],
    '95% upper': [np.percentile(bootstrap_auc, 97.5), np.percentile(bootstrap_ap, 97.5)]
}, index=['ROC-AUC', 'Average precision'])
display(intervals.round(4))
print('Event rate:', round(base_rate, 4))

,Estimate,95% lower,95% upper
ROC-AUC,0.8245,0.8192,0.8299
Average precision,0.4593,0.4456,0.4714


Event rate: 0.1576


## 4. Describe The Split

I used a stratified random split to approximate predicting recorded status for similar unseen survey respondents from the same survey setting, while noting that this does not test future populations or verify separation by person.

I kept 64% for training, 16% for validation, and 20% as an unused final test. Identical answers do not prove that rows belong to the same person. The prepared file has no respondent ID, so person overlap cannot be checked.

## Summary

The original validation AUC was **0.8245**, with a row-bootstrap 95% interval of **0.8192–0.8299**. Shuffling the labels gave AUC **0.4985**, close to chance. Average precision was **0.4593**, compared with an event rate of **0.1576** (15.76%); its row-bootstrap interval was **0.4456–0.4714**. This shuffled result is reassuring, but one run cannot prove that all leakage is absent. The intervals apply to this fixed model and validation sample.

**Limitation:** The requested patient-level bootstrap cannot be completed with this prepared file because it has no patient ID. I reported a row-based interval and clearly labelled that limitation.

**Data:** https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset  
**Average precision:** https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html  